<a href="https://colab.research.google.com/github/Kripton2005/larp_larp_larp/blob/main/algoarena.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting a synthetic stock from news headlines

Making money in trading is about being fast and smart. Excellent coding covers the "fast" part, but how can we be "smart"?

The goal of Algo is to make **hypotheses** about the world and to be able to **evaluate** them. Whenever we have a hypothesis, the first step is almost always going to be to look at the data.

In this challenge, the market is simulated. Each **session** is one synthetic stock trading for 100 time **bars**. Along the way, news headlines come out. The headlines are fictional and so are the companies they mention.

You see the first half of each session: bars 0 to 49 and the headlines published during them. At the close of bar 49 you take a position, and you hold it for 10 bars, until the close of bar 59. Your job is to predict the stock's return over those 10 bars.

The returns are **market-adjusted**: we subtract the average return across all sessions, so a session's return is positive if it did better than a typical session. Betting that every stock goes up earns nothing here.

## Setup
You should make a copy of this notebook. The first cell will download the data and source needed.

In [ ]:
from importlib.util import find_spec
from pathlib import Path
import subprocess
import sys

# The official data loader reads parquet files.
if find_spec("pyarrow") is None and find_spec("fastparquet") is None:
    import numpy as np
    arrow = "pyarrow>=17,<19" if int(np.__version__.split(".")[0]) < 2 else "pyarrow>=17"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", arrow], check=True)

missing = [name for name in ("scipy", "attrs", "rich", "tqdm", "ipywidgets") if find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

FOLDER_URL = "https://drive.google.com/drive/folders/1mYOtE9R-XtdcJ4cPVpQYAxiPLD1Dm4vX?usp=sharing"
data_files = (
    "sessions.py", "ins_bars.parquet", "ins_headlines.parquet", "ins_returns.parquet",
    "oos_bars.parquet", "oos_headlines.parquet",
)
data_dir = next(
    (folder for folder in (Path.cwd(), Path.cwd() / "content") if all((folder / name).is_file() for name in data_files)),
    None,
)
if data_dir is None:
    data_dir = Path.cwd()
    if find_spec("gdown") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    gdown.download_folder(url=FOLDER_URL, output=str(data_dir))
sys.path.insert(0, str(data_dir))

In [ ]:
import sessions # pyright: ignore[reportMissingImports]

# Part 1. Data exploration

The first section of this notebook loads the data and does some basic exploration. You should run each cell. Feel free to add and run additional cells to explore the data further.

## Get the data

You have 10,000 in-sample sessions for training and 10,000 out-of-sample sessions to predict.

Make sure the `ins_*.parquet` and `oos_*.parquet` files are in the same directory as this notebook and `sessions.py`.

In [ ]:
df = sessions.get_ins_data()

In [ ]:
df.head()

Each row is one in-sample session:

- `open`, `high`, `low`, `close`: arrays of 50 prices, one per bar from 0 to 49. Prices start at 1.
- `headlines`: the headlines published during bars 0 to 49, in order.
- `headline_bars`: the bar each of those headlines was published in.
- `returns`: the market-adjusted return from the close of bar 49 to the close of bar 59. This is what you predict.

## What does a session look like?

In [ ]:
session = df.iloc[0]
for bar, headline in zip(session["headline_bars"], session["headlines"], strict=True):
    print(bar, headline)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(session["close"])
plt.xlabel("Bar")
plt.ylabel("Close price")

## Visualize the distribution of returns

In [ ]:
df["returns"].hist(bins=50)

## The whole session, for research

In-sample, you can also see the second half of every session: bars 50 to 99 and their headlines. Out-of-sample you will only have the first half, so your prediction can't use these. They are still useful for research, for example to look at what prices do after a headline.

In [ ]:
full_bars = sessions.get_ins_full_bars()
full_headlines = sessions.get_ins_full_headlines()
full_headlines.head()

# Part 2. Predicting returns

The rest of this notebook is dedicated to the actual task of predicting returns. It explains the task and evaluation, and also gives some basic ideas and useful utility functions.

## Your task

Your goal is to write a function that takes a row of `df` (minus the `returns` column) and returns a prediction of the return from the close of bar 49 to the close of bar 59.
Here is an example of a valid submission, based on the naive idea that "stocks go up".

```python
def my_prediction(row: pd.Series) -> float:
    return 1.0
```

You can call `sessions.get_ins_predictions` to see your predictions on the in-sample dataset: this will be provided as the new column `"pred"` in your dataframe.

In [ ]:
def my_prediction(row):
    return 1.0  # stocks go up!

In [ ]:
df_predictions = sessions.get_ins_predictions(my_prediction)
df_predictions[["headlines", "returns", "pred"]].head()

## How are you evaluated?

Your goal is to produce a prediction that makes money AS CONSISTENTLY AS POSSIBLE.
We have implemented a very simple trader that will use your prediction to trade. Your score is the *Sharpe ratio* of this trader's performance.

Here is how the trader behaves:
* At the close of bar 49, it takes a position equal to your prediction: it buys if your prediction is positive and sells if it is negative, more for a larger prediction.
* It holds the position until the close of bar 59 and receives `prediction * return`.
* Each session counts as one trading day, so the Sharpe ratio is `sqrt(252) * mean(session_profit) / stddev(session_profit)`. This is a common metric we use in Algo to evaluate our strategies.

## Running an in-sample backtest

We can use your model's in-sample performance to *estimate* how well it will perform out-of-sample. We use these in-sample *backtests* to evaluate our trading hypotheses. The function `sessions.run_ins_backtest` calls your prediction model in-sample and computes the Sharpe ratio (as well as several other metrics).

In [ ]:
sessions.run_ins_backtest(my_prediction)

Because returns are market-adjusted, "stocks go up" has a Sharpe ratio of about zero. You'll have to do better!

Here's how to read these stats:

- **Sharpe** is the in-sample Sharpe ratio of your prediction model.
- **Total PnL** is the amount of money that your strategy made.
- **Frac nonzero prediction** is the fraction of sessions where your model issued a nonzero prediction.
- **Correlation** is the correlation between your prediction and the returns.
- **Mean return positive** is the average return among sessions for which you issued a positive forecast. Ideally, this number would be positive.
- **Mean return negative** is the average return among sessions for which you issued a negative forecast. Ideally, this number would be negative.
- **Mean daily portfolio size** is the average size of your position. Roughly, it measures the amount of capital you need to actually implement your strategy.
- **Annual return** is the ratio of your total PnL to the total size of your positions. In practice, a strategy that makes a lot of money (total PnL) but has a low annual return is hard to implement due to the large capital requirements.

## Getting ready to go out-of-sample

Once you think you have a good submission, you can run it on the out-of-sample sessions. These are the predictions you will submit!

In [ ]:
df_oos_predictions = sessions.get_oos_predictions(my_prediction)
df_oos_predictions.head()

### Scoring

How good is your model? Well, you can't run a backtest, because the out-of-sample returns are secret!

Instead, `sessions.write_submission(...)` writes your out-of-sample predictions to `predictions.csv`. Arcade calls its columns `headline_id,pred`; `headline_id` holds the session number. **Upload that file on the challenge page in arcade**, which scores it against the hidden out-of-sample returns. Your out-of-sample Sharpe appears on the live leaderboard.

Be aware that you only get **10 uploads**, so don't waste them! Use in-sample backtests to decide what is worth submitting.

In [ ]:
# This will not work!
# sessions.run_oos_backtest(my_prediction)

## Ideas to get you started

Here are some ideas you may wish to explore in your submission. Of course, you are not required to pursue these ideas.

**Looking at keywords**. Some headlines sound like good news and some sound like bad news. Maybe a session with more good news does better?

In [ ]:
def keyword_predictor(row):
    score = 0.0
    for headline in row["headlines"]:
        if "secures" in headline:
            score += 1.0
        if "decline" in headline:
            score -= 1.0
    return score

In [ ]:
sessions.run_ins_backtest(keyword_predictor)

# Fitting headline effects

The training sessions show that prices keep moving for about ten bars after a headline. A report at bar 48 therefore carries more information about bars 50–59 than the same report at bar 40. We estimate a separate response for each headline template from the full training price paths. Scaling price moves by each session’s average high–low range lets us compare reactions across stocks with different volatility.

The data cells below use one company slot for each distinct name mentioned. Sessions with few headlines can leave slots empty.

In [ ]:
from dataclasses import dataclass
from functools import lru_cache
from math import perm
import re

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import minimize_scalar
from scipy.signal import lfilter
from scipy.special import logsumexp, softmax


# Each suffix identifies an industry in the training headlines.
INDUSTRIES = [
    ('Commerce', 'Marketplace', 'Retail Group', 'Outlets', 'Brands', 'Stores', 'Trading Co', 'Goods'),
    ('Systems', 'Microchips', 'Labs', 'Computing', 'Software', 'Networks', 'Technologies', 'Devices'),
    ('Sciences', 'Genomics', 'Diagnostics', 'Biotech', 'Pharmaceuticals', 'Therapeutics', 'Biopharma', 'Biosciences'),
    ('Securities', 'Capital', 'Investments', 'Holdings', 'Asset Group', 'Financial', 'Partners', 'Advisors'),
    ('Fuels', 'Power', 'Resources', 'Grid', 'Renewables', 'Solutions', 'Energy', 'Industries'),
]
INDUSTRY = {name: i for i, names in enumerate(INDUSTRIES) for name in names}
# A headline falls inside its publication bar. The two endpoints each receive half a bar.
KERNEL = np.r_[0.5, np.ones(9), 0.5]
HEADLINE = re.compile(r'^(.+?) (\b(?:[a-z][a-z-]*|CEO|CFO|CTO|COO|Chief)\b.*)')
NUMBER = re.compile(r'\$?(\d+(?:\.\d+)?)[BM%]?')


@lru_cache(maxsize=50000)
def parse_headline(text):
    company, body = HEADLINE.fullmatch(text).groups()
    body = re.sub(r'^.*?(steps down|addresses investor)', r'\1', body)
    number = NUMBER.search(body)
    words = NUMBER.sub('#', body).split()
    length = 3 if words[:2] in (['reports', '#'], ['sees', '#']) else 2
    value = np.log1p(float(number[1])) if number else 0.0
    return company, ' '.join(words[:length]), value


@dataclass
class Panel:
    ids: pd.Index
    price: np.ndarray      # Log close in units of the session's mean log bar range
    scale: np.ndarray
    sectors: np.ndarray    # One slot per distinct company, in order of first appearance
    events: pd.DataFrame


def make_panel(ids, close, high, low, headlines):
    ids = pd.Index(ids, name='session')
    scale = np.log(high / low).mean(axis=1)
    events = headlines[['session', 'bar_ix']].copy()
    events['row'] = ids.get_indexer(events.session)
    parsed = [parse_headline(text) for text in headlines.headline]
    events[['company', 'topic', 'number']] = pd.DataFrame(
        parsed, columns=['company', 'topic', 'number'], index=events.index
    )
    companies = events[['row', 'company']].drop_duplicates().copy()
    companies['slot'] = companies.groupby('row').cumcount()
    sectors = np.full((len(ids), 4), -1, dtype=int)
    suffix = companies.company.str.split(n=1).str[1]
    sectors[companies.row, companies.slot] = suffix.map(INDUSTRY).to_numpy(dtype=int)
    events = events.merge(companies, on=['row', 'company'], validate='many_to_one')
    return Panel(ids, np.log(close) / scale[:, None], scale, sectors, events)


def training_panel(bars, headlines, ids):
    prices = bars[bars.session.isin(ids)].set_index(['session', 'bar_ix']).unstack().reindex(ids)
    return make_panel(ids, prices.close.to_numpy(), prices.high.to_numpy(), prices.low.to_numpy(),
                      headlines[headlines.session.isin(ids)])


def visible_panel(view):
    headlines = (view[['headlines', 'headline_bars']].explode(['headlines', 'headline_bars'])
                 .dropna().rename(columns={'headlines': 'headline', 'headline_bars': 'bar_ix'})
                 .reset_index())
    return make_panel(view.index, np.stack(view.close), np.stack(view.high), np.stack(view.low), headlines)

## Learning which headlines move the stock

The names that appear together suggest a main industry, a focal company and industry peers. The model assigns probabilities to the possible focal companies, including a company that has yet to appear. It learns how much more strongly prices react to the focal company by comparing candidate weights against observed training prices.

The regression fits the signed response to each template, with a separate, more strongly regularized coefficient for the number in its text. It also estimates how the current price relates to the next bar’s return.

In [ ]:
def company_scenarios(sectors):
    # For each possible main sector and focal company, count the compatible sets of four names.
    # Three names come from one of the five 16-company sectors. The fourth can be in any sector.
    n = len(sectors)
    sector_match = sectors[:, None, :] == np.arange(len(INDUSTRIES))[None, :, None]
    members = sector_match.sum(axis=2)
    outsiders = (sectors >= 0).sum(axis=1)[:, None] - members
    prior = np.zeros((n, len(INDUSTRIES), 5))  # Four visible focal slots and one unobserved focal
    for total, chance in [(3, 64 / 77), (4, 13 / 77)]:
        coverage = np.array([perm(total, k) / perm(16, k) for k in range(5)])[members]
        other_coverage = np.where(outsiders == 0, 1.0, (outsiders == 1) * (4 - total) / 64)
        mass = chance * coverage * other_coverage
        prior[:, :, :4] += mass[:, :, None] * sector_match / total
        prior[:, :, 4] += mass * (1 - members / total)
    prior = prior.reshape(n, -1)
    prior /= prior.sum(axis=1, keepdims=True)
    log_prior = np.full_like(prior, -np.inf)
    np.log(prior, out=log_prior, where=prior > 0)

    sector = np.broadcast_to(sector_match[:, :, None, :], (n, 5, 5, 4)).reshape(n, 25, 4)
    focal = np.tile(np.eye(5, 4), (5, 1))  # The last row represents an unobserved company.
    return log_prior, sector, focal


def company_weights(scenarios, focal_ratio):
    _, sector, focal = scenarios
    # The focal company and its two peers have average weight one.
    return 3 / (2 + focal_ratio) * (sector + (focal_ratio - 1) * focal)


@dataclass
class NewsModel:
    topics: pd.Index
    number_means: np.ndarray
    effects: np.ndarray
    number_effects: np.ndarray
    intercept: float
    price_effect: float


def fit_headlines(panel, company_weight):
    n, bars = panel.price.shape
    e = panel.events
    topics = pd.Index(sorted(e.topic.unique()))
    topic = topics.get_indexer(e.topic)
    means = e.groupby('topic').number.mean().reindex(topics).to_numpy(dtype=float)
    number = e.number.to_numpy(dtype=float) - means[topic]
    row, bar, slot = (e[col].to_numpy(dtype=int) for col in ('row', 'bar_ix', 'slot'))
    weight = company_weight[row, slot]
    rows, cols, values, amounts = [], [], [], []
    for lag, fraction in enumerate(KERNEL):
        use = bar + lag < bars
        rows.append(row[use] * bars + bar[use] + lag)
        cols.append(topic[use])
        values.append(fraction * weight[use])
        amounts.append(fraction * weight[use] * number[use])
    indices = (np.concatenate(rows), np.concatenate(cols))
    shape = (n * bars, len(topics))
    counts = sparse.coo_matrix((np.concatenate(values), indices), shape=shape)
    numbers = sparse.coo_matrix((np.concatenate(amounts), indices), shape=shape)
    previous = np.pad(panel.price[:, :-1], ((0, 0), (1, 0)))
    design = sparse.hstack((np.ones((n * bars, 1)), previous.reshape(-1, 1), counts, numbers), format='csr')
    changes = (panel.price - previous).ravel()
    penalty = np.r_[0, 0, np.full(len(topics), 60), np.full(len(topics), 2200)]
    coef = np.linalg.solve((design.T @ design).toarray() + np.diag(penalty), design.T @ changes)
    return NewsModel(topics, means, coef[2:2 + len(topics)], coef[2 + len(topics):], coef[0], coef[1])


def news_path(panel, news_model):
    # The extra ten columns contain only the remaining impact of headlines already visible.
    e = panel.events
    topic = news_model.topics.get_indexer(e.topic)
    known = topic >= 0
    topic = topic[known]
    number = e.loc[known, 'number'].to_numpy(dtype=float) - news_model.number_means[topic]
    impact = news_model.effects[topic] + news_model.number_effects[topic] * number
    impulses = np.zeros((len(panel.ids), 4, panel.price.shape[1] + 10))
    coords = tuple(e.loc[known, col].to_numpy(dtype=int) for col in ('row', 'slot', 'bar_ix'))
    np.add.at(impulses, coords, impact)
    return lfilter(KERNEL, [1.0], impulses, axis=-1)


def price_evidence(panel, news_model, path):
    # These cross-products score every possible focal company against the observed price path.
    previous = np.pad(panel.price[:, :-1], ((0, 0), (1, 0)))
    observed = panel.price - previous - news_model.intercept - news_model.price_effect * previous
    past = path[:, :, :panel.price.shape[1]]
    linear = np.einsum('nct,nt->nc', past, observed)
    quadratic = np.einsum('nct,ndt->ncd', past, past)
    return linear, quadratic, observed


def scenario_log_likelihood(evidence, weight, noise_variance):
    linear, quadratic, _ = evidence
    cross = np.einsum('nsc,nc->ns', weight, linear)
    squared = np.einsum('nsc,ncd,nsd->ns', weight, quadratic, weight)
    return (cross - 0.5 * squared) / noise_variance

## Forecasting and sizing the position

At bar 49, the observed price reactions update the probabilities of the candidate companies. For each candidate, we sum the remaining impact of published headlines over the next ten bars. The probability-weighted average gives the forecast, and the spread across candidates contributes to its uncertainty.

We subtract the cross-session mean return and size the position using the forecast divided by the estimated second moment of the return. This accounts for how much each position contributes to the variability of the profits that Arcade scores.

In [ ]:
@dataclass
class ForecastModel:
    news: NewsModel
    focal_ratio: float
    noise_variance: float
    future_drift: float = 0.0
    holding_variance: float = 1.0


def forecast_components(panel, model, *, return_history=False):
    scenarios = company_scenarios(panel.sectors)
    weight = company_weights(scenarios, model.focal_ratio)
    path = news_path(panel, model.news)
    evidence = price_evidence(panel, model.news, path)
    posterior = softmax(scenarios[0] + scenario_log_likelihood(evidence, weight, model.noise_variance), axis=1)
    carry = (1 + model.news.price_effect) ** np.arange(9, -1, -1)
    remaining = path[:, :, -10:] @ carry
    moves = np.einsum('nsc,nc->ns', weight, remaining)
    expected = (posterior * moves).sum(axis=1)
    variance = (posterior * (moves - expected[:, None]) ** 2).sum(axis=1)
    drift = (model.news.intercept + model.news.price_effect * panel.price[:, -1]) * carry.sum()
    if return_history:
        weights = np.einsum("ns,nsc->nc", posterior, weight)
        history = np.einsum("nc,nct->nt", weights, path[:, :, :-10])
        prior_mean = (np.exp(scenarios[0]) * moves).sum(axis=1)
        return expected + drift, variance, expected - prior_mean, evidence[2] - history, history, weights
    return expected + drift, variance


def fit_model(bars, headlines, training_view):
    full = training_panel(bars, headlines, training_view.index)
    scenarios = company_scenarios(full.sectors)
    prior = np.exp(scenarios[0])
    weight = company_weights(scenarios, 4)
    average_weight = np.einsum('ns,nsc->nc', prior, weight)
    news = fit_headlines(full, average_weight)
    path = news_path(full, news)
    evidence = price_evidence(full, news, path)
    mean_path = np.einsum('nc,nct->nt', average_weight, path[:, :, :-10])
    noise_variance = np.mean((evidence[2] - mean_path) ** 2)

    def loss(ratio):
        weight = company_weights(scenarios, ratio)
        scores = scenario_log_likelihood(evidence, weight, noise_variance)
        return -logsumexp(scenarios[0] + scores, axis=1).mean()

    ratio = minimize_scalar(loss, bounds=(1, 9), method='bounded').x
    model = ForecastModel(news, ratio, noise_variance)
    visible = visible_panel(training_view)
    mean, uncertainty = forecast_components(visible, model)
    observed_log_move = (full.price[:, 59] - full.price[:, 49]) * full.scale
    model.future_drift = np.mean(observed_log_move / visible.scale - mean)
    raw = np.expm1((mean + model.future_drift) * visible.scale)
    residual = training_view.returns.to_numpy() - (raw - raw.mean())
    model.holding_variance = np.mean((residual / visible.scale) ** 2 - uncertainty)
    return model


def predict_sessions(view, model):
    panel = visible_panel(view)
    mean, uncertainty = forecast_components(panel, model)
    raw = np.expm1((mean + model.future_drift) * panel.scale)
    forecast = raw - raw.mean()
    second_moment = (model.holding_variance + uncertainty) * panel.scale ** 2 + forecast ** 2
    position = forecast * second_moment.mean() / second_moment
    return pd.DataFrame({'forecast': forecast, 'pred': position}, index=view.index)

## Calibrate returns and position sizes

The first model estimates how prices respond to news. The second stage measures whether the past price moves that the news leaves unexplained help forecast the next ten bars. It also measures risk using opening gaps, candle bodies, and the distance from each body to the bar's high and low.

Each training session supplies forecasts at 15 cutoffs between bars 19 and 89. At each cutoff, the features use only prices and published headlines through that bar. The next ten bars supply the training outcome. Entire sessions are held out together for evaluation.

In [ ]:
@dataclass
class RidgeCorrection:
    center: np.ndarray
    scale: np.ndarray
    coefficients: np.ndarray

    @classmethod
    def fit(cls, x, y):
        center = x.mean(axis=0)
        scale = np.maximum(x.std(axis=0), 1e-8)
        design = np.column_stack((np.ones(len(x)), (x - center) / scale))
        penalty = np.diag(np.r_[0.0, np.full(x.shape[1], 0.04 * len(x))])
        coefficients = np.linalg.solve(design.T @ design + penalty, design.T @ y)
        return cls(center, scale, coefficients)

    def predict(self, x):
        return self.coefficients[0] + ((x - self.center) / self.scale) @ self.coefficients[1:]


def calibration_inputs(panel, model, log_open, log_high, log_low):
    mean, uncertainty, role_update, residual, news_history, weights = forecast_components(
        panel, model, return_history=True
    )
    n, end = panel.price.shape
    scale = panel.scale
    raw = np.expm1((mean + model.future_drift) * scale) / scale
    events = panel.events
    counts = np.bincount(events.row.to_numpy(dtype=int), minlength=n)

    # Does the part of a recent news move already observed change its remaining effect?
    reaction = np.zeros(n)
    for age in range(10):
        arrivals = events.loc[events.bar_ix == end - 1 - age]
        rows = arrivals.row.to_numpy(dtype=int)
        slots = arrivals.slot.to_numpy(dtype=int)
        move_so_far = residual[:, -(age + 1):] @ KERNEL[:age + 1]
        np.add.at(reaction, rows, weights[rows, slots] * move_so_far[rows] * KERNEL[age + 1:].sum())

    mean_x = np.column_stack((
        raw, role_update, panel.price[:, -1], news_history.sum(axis=1),
        residual[:, -3:].sum(axis=1), residual[:, -10:].sum(axis=1),
        counts, scale, np.full(n, end), reaction,
    ))
    log_close = panel.price * scale[:, None]
    gap = (log_open[:, 1:] - log_close[:, :-1]) / scale[:, None]
    body = (log_close - log_open) / scale[:, None]
    upper = (log_high - np.maximum(log_open, log_close)) / scale[:, None]
    lower = (np.minimum(log_open, log_close) - log_low) / scale[:, None]
    ranges = (log_high - log_low) / scale[:, None]
    risk_x = np.column_stack((
        (gap**2).mean(axis=1), (body**2).mean(axis=1), (upper**2 + lower**2).mean(axis=1),
        (residual**2).mean(axis=1), (ranges[:, -20:]**2).mean(axis=1),
        (ranges[:, -5:]**2).mean(axis=1), uncertainty, counts, scale, np.full(n, end),
    ))
    return mean_x, risk_x, raw, uncertainty


def fit_calibration(bars, headlines, training_ids, model):
    # Each label is ten bars ahead. For each cutoff, construct its inputs from the prefix alone.
    ordered = bars[bars.session.isin(training_ids)].set_index(["session", "bar_ix"]).unstack().reindex(training_ids)
    op, hi, lo, cl = (np.log(ordered[k].to_numpy()) for k in ("open", "high", "low", "close"))
    full = make_panel(training_ids, ordered.close.to_numpy(), ordered.high.to_numpy(), ordered.low.to_numpy(),
                      headlines.loc[headlines.session.isin(training_ids)])
    observations = []
    for end in range(20, 91, 5):
        scale = (hi[:, :end] - lo[:, :end]).mean(axis=1)
        visible = full.events.loc[full.events.bar_ix < end]
        sectors = np.full_like(full.sectors, -1)
        r, c = visible.row.to_numpy(dtype=int), visible.slot.to_numpy(dtype=int)
        sectors[r, c] = full.sectors[r, c]
        panel = Panel(full.ids, cl[:, :end] / scale[:, None], scale, sectors, visible)
        mx, vx, raw, uncertainty = calibration_inputs(panel, model, op[:, :end], hi[:, :end], lo[:, :end])
        realized = np.expm1(cl[:, end + 9] - cl[:, end - 1])
        observations.append((mx, vx, raw, uncertainty, scale, realized))

    mx, vx, raw, uncertainty, scale, realized = (np.concatenate(col) for col in zip(*observations))
    mean_model = RidgeCorrection.fit(mx, realized / scale - raw)
    raw_forecast = (raw + mean_model.predict(mx)) * scale
    # Market adjustment is separate for each training cutoff.
    forecast = raw_forecast.reshape(len(observations), -1)
    actual = realized.reshape(len(observations), -1)
    errors = (actual - actual.mean(axis=1, keepdims=True)) - (forecast - forecast.mean(axis=1, keepdims=True))
    risk_model = RidgeCorrection.fit(vx, (errors.ravel() / scale) ** 2 - uncertainty)
    return mean_model, risk_model


def refined_predictions(visible_view, model, calibration):
    panel = visible_panel(visible_view)
    op, hi, lo = (np.log(np.stack(visible_view[k])) for k in ("open", "high", "low"))
    mx, vx, raw, uncertainty = calibration_inputs(panel, model, op, hi, lo)
    mean_model, risk_model = calibration
    raw_return = (raw + mean_model.predict(mx)) * panel.scale
    forecast = raw_return - raw_return.mean()
    variance = np.clip(risk_model.predict(vx), 0.55 * model.holding_variance, 1.65 * model.holding_variance)
    second_moment = (variance + uncertainty) * panel.scale**2 + forecast**2
    position = forecast * second_moment.mean() / second_moment
    return pd.DataFrame({"forecast": forecast, "pred": position}, index=panel.ids)

## Held-out backtest

The fixed split reserves 2,000 entire sessions. All parameter fitting, including risk calibration, uses the other 8,000. Each held-out prediction sees the same 50 bars and headlines that will be available out of sample.

In [ ]:
def backtest_row(position, actual):
    pnl = position * actual
    return {
        "Sharpe": np.sqrt(252) * pnl.mean() / (pnl.std(ddof=1) + 1e-12),
        "Correlation": position.corr(actual),
    }


order = np.random.default_rng(42).permutation(len(df))
training = df.iloc[order[:8000]]
held_out = df.iloc[order[8000:]]
validation_model = fit_model(full_bars, full_headlines, training)
visible = held_out.drop(columns="returns")
comparison = predict_sessions(visible, validation_model)
comparison["Keyword rule"] = visible.apply(keyword_predictor, axis=1)
comparison = comparison.rename(columns={"forecast": "Return forecast", "pred": "Sized position"})
calibration = fit_calibration(full_bars, full_headlines, training.index, validation_model)
comparison["Calibrated position"] = refined_predictions(visible, validation_model, calibration)["pred"]
validation_scores = pd.DataFrame({
    name: backtest_row(comparison[name], held_out["returns"])
    for name in ["Keyword rule", "Return forecast", "Sized position", "Calibrated position"]
}).T
validation_scores

## Write the calibrated submission

Refit both stages on all 10,000 labeled sessions. The new CSV is `predictions_improved.csv`. The original team submission is `predictions.csv`.

In [ ]:
model = fit_model(full_bars, full_headlines, df)
oos = sessions.get_oos_data()
calibration = fit_calibration(full_bars, full_headlines, df.index, model)
predictions = refined_predictions(oos, model, calibration)[["pred"]]
assert predictions.index.is_unique and predictions.index.equals(oos.index)
assert np.isfinite(predictions["pred"]).all()
submission_path = sessions.write_submission(prediction_df=predictions, path="predictions_improved.csv")

written = pd.read_csv(submission_path)
assert written.columns.tolist() == ["headline_id", "pred"]
assert len(written) == len(oos) == 10_000
np.testing.assert_array_equal(written["headline_id"], oos.index)
np.testing.assert_allclose(written["pred"], predictions["pred"], rtol=1e-12, atol=1e-15)
written.head()